In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

In [3]:
print("Fetching Online Retail dataset from the UCI...")
online_retail = fetch_ucirepo(id=352)
online_retail

Fetching Online Retail dataset from the UCI...


{'data': {'ids':        InvoiceNo StockCode
  0         536365    85123A
  1         536365     71053
  2         536365    84406B
  3         536365    84029G
  4         536365    84029E
  ...          ...       ...
  541904    581587     22613
  541905    581587     22899
  541906    581587     23254
  541907    581587     23255
  541908    581587     22138
  
  [541909 rows x 2 columns],
  'features':                                 Description  Quantity      InvoiceDate  \
  0        WHITE HANGING HEART T-LIGHT HOLDER         6   12/1/2010 8:26   
  1                       WHITE METAL LANTERN         6   12/1/2010 8:26   
  2            CREAM CUPID HEARTS COAT HANGER         8   12/1/2010 8:26   
  3       KNITTED UNION FLAG HOT WATER BOTTLE         6   12/1/2010 8:26   
  4            RED WOOLLY HOTTIE WHITE HEART.         6   12/1/2010 8:26   
  ...                                     ...       ...              ...   
  541904          PACK OF 20 SPACEBOY NAPKINS        12  12/9

In [7]:
X = online_retail.data.features
X.head()

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [8]:
ids = online_retail.data.ids
ids.head()

,InvoiceNo,StockCode
0,536365,85123A
1,536365,71053
2,536365,84406B
3,536365,84029G
4,536365,84029E


In [9]:
df = ids.join(X)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [10]:
print(f"Raw shape: {df.shape}")
print("Missing values per column:\n", df.isnull().sum(), sep="")

Raw shape: (541909, 8)
Missing values per column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## Cleaning

In [12]:
# drop rows without CustomerID
df = df.dropna(subset=["CustomerID"]).copy()

# Fill missing product descriptions
df["Description"] = df["Description"].fillna("Unknown Product")

# Remove duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows removed: {dup_count}")
df = df.drop_duplicates().reset_index(drop=True)

Duplicate rows removed: 5225


In [14]:
# Drop non-product / administratice StockCodes
#   (postage, manual adjustments, bank_charges, amazon fees, bad debt, etc)

bad_codes = {"POST", "D", "C2", "M", "BANK CHARGES", "AMAZONFEE", "DOT", "CRUK", "PADS", "B"}
df = df[~df['StockCode'].str.upper().isin(bad_codes)]

In [15]:
## Cancellations: InvoiceNo starting with 'C' = cancelled order
# Keep them in a separate frame for return behaviour features, then
# drop from the main transaction frame

df["InvoiceNo"] = df["InvoiceNo"].astype(str)
cancellations = df[df['InvoiceNo'].str.startswith("C")].copy()
df = df[~df['InvoiceNo'].str.startswith("C")]

In [16]:
# keep only strictly positive quantities and prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)

# Derived cols
df['Amount'] = df['Quantity'] * df['UnitPrice']
df['Date']   = df['InvoiceDate'].dt.date
df['Time']   = df['InvoiceDate'].dt.time
df['Hour']   = df['InvoiceDate'].dt.hour
df['DOW']    = df['InvoiceDate'].dt.day_name()
df['month']  = df['InvoiceDate'].dt.month

print(f"Clean transaction shape: {df.shape}")
print(f"Date range: {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Unique invoices : {df['InvoiceNo'].nunique()}")
print(f"Countries       : {df['Country'].nunique()}")

Clean transaction shape: (391150, 14)
Date range: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00
Unique customers: 4334
Unique invoices : 18402
Countries       : 37


## Feature Engineering

In [17]:
## Reference data = day after last transaction, the standard RFM conversion
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

snapshot_date

Timestamp('2011-12-10 12:50:00')

In [18]:
# RFM core
rfm = df.groupby("CustomerID").agg(
    Recency  = ("InvoiceDate", lambda s: (snapshot_date - s.max()).days),
    Frequency = ("InvoiceNo", "nunique"),
    Monetary  = ("Amount", "sum"),
)

rfm

,Recency,Frequency,Monetary
CustomerID,,,
12346,326,1,77183.60
12347,2,7,4310.00
12348,75,4,1437.24
12349,19,1,1457.55
12350,310,1,294.40
...,...,...,...
18280,278,1,180.60
18281,181,1,80.82
18282,8,2,178.05


In [24]:
# extra behavioral features usefil for segmentation beyon pure RFM
extras = df.groupby("CustomerID").agg(
    TotalQuantity       = ("Quantity",  "sum"),
    UniqueProducts      = ("StockCode", "nunique"),
    AvgUnitPrice        = ("UnitPrice", "mean"),
    AvgBasketValue      = ("Amount",    "mean"),
    FirstPurchase       = ("InvoiceDate", "min"),
    LastPurchase        = ("InvoiceDate", "max")
)
extras['TenureDays'] = (extras['LastPurchase'] - extras['FirstPurchase'])
extras

,TotalQuantity,UniqueProducts,AvgUnitPrice,AvgBasketValue,FirstPurchase,LastPurchase,TenureDays
CustomerID,,,,,,,
12346,74215,1,1.040000,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00,0 days 00:00:00
12347,2458,103,2.644011,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00,365 days 00:55:00
12348,2332,21,0.692963,53.231111,2010-12-16 19:09:00,2011-09-25 13:13:00,282 days 18:04:00
12349,630,72,4.237500,20.243750,2011-11-21 09:51:00,2011-11-21 09:51:00,0 days 00:00:00
12350,196,16,1.581250,18.400000,2011-02-02 16:01:00,2011-02-02 16:01:00,0 days 00:00:00
...,...,...,...,...,...,...,...
18280,45,10,4.765000,18.060000,2011-03-07 09:52:00,2011-03-07 09:52:00,0 days 00:00:00
18281,54,7,5.622857,11.545714,2011-06-12 10:53:00,2011-06-12 10:53:00,0 days 00:00:00
18282,103,12,5.199167,14.837500,2011-08-05 13:35:00,2011-12-02 11:43:00,118 days 22:08:00


In [22]:
# Average order value (per invoice) and items per order
order_totals = df.groupby(['CustomerID', 'InvoiceNo']).agg(
    order_value = ("Amount", "sum"),
    order_items = ("Quantity", "sum"),
).reset_index()

order_totals

,CustomerID,InvoiceNo,order_value,order_items
0,12346,541431,77183.60,74215
1,12347,537626,711.79,319
2,12347,542237,475.39,315
3,12347,549222,636.25,483
4,12347,556201,382.52,196
...,...,...,...,...
18397,18283,579673,220.31,132
18398,18283,580872,208.00,142
18399,18287,554065,765.28,488
18400,18287,570715,1001.32,990


In [23]:
per_order = order_totals.groupby("CustomerID").agg(
    AvgOrderValue = ("order_value", "mean"),
    AvgOrderItems = ("order_items", "mean"),
)

per_order

,AvgOrderValue,AvgOrderItems
CustomerID,,
12346,77183.600000,74215.000000
12347,615.714286,351.142857
12348,359.310000,583.000000
12349,1457.550000,630.000000
12350,294.400000,196.000000
...,...,...
18280,180.600000,45.000000
18281,80.820000,54.000000
18282,89.025000,51.500000


In [25]:
cancellations["CustomerID"] = pd.to_numeric(
    cancellations["CustomerID"], errors="coerce"
)

returns = (
    cancellations.dropna(subset=['CustomerID'])
    .assign(CustomerID=lambda d: d["CustomerID"].astype(int))
    .groupby("CustomerID")
    .agg(ReturnInvoices=("InvoiceNo", "nunique"))
)

returns

,ReturnInvoices
CustomerID,
12346,1
12352,1
12359,2
12362,3
12375,1
...,...
18272,1
18274,1
18276,2


In [26]:
# Dominant country per customer (mode)
country = (
    df.groupby("CustomerID")["Country"]
    .agg(lambda s: s.mode().iat[0])
    .rename("Country")
)
country

CustomerID
12346    United Kingdom
12347           Iceland
12348           Finland
12349             Italy
12350            Norway
              ...      
18280    United Kingdom
18281    United Kingdom
18282    United Kingdom
18283    United Kingdom
18287    United Kingdom
Name: Country, Length: 4334, dtype: str

In [28]:
# Assemble
customer_features = (
    rfm
    .join(extras)
    .join(per_order)
    .join(returns)
    .join(country)
)

customer_features.head()

,Recency,Frequency,Monetary,TotalQuantity,UniqueProducts,AvgUnitPrice,AvgBasketValue,FirstPurchase,LastPurchase,TenureDays,AvgOrderValue,AvgOrderItems,ReturnInvoices,Country
CustomerID,,,,,,,,,,,,,,
12346,326,1,77183.60,74215,1,1.040000,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00,0 days 00:00:00,77183.600000,74215.000000,1.0,United Kingdom
12347,2,7,4310.00,2458,103,2.644011,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00,365 days 00:55:00,615.714286,351.142857,NaN,Iceland
12348,75,4,1437.24,2332,21,0.692963,53.231111,2010-12-16 19:09:00,2011-09-25 13:13:00,282 days 18:04:00,359.310000,583.000000,NaN,Finland
12349,19,1,1457.55,630,72,4.237500,20.243750,2011-11-21 09:51:00,2011-11-21 09:51:00,0 days 00:00:00,1457.550000,630.000000,NaN,Italy
12350,310,1,294.40,196,16,1.581250,18.400000,2011-02-02 16:01:00,2011-02-02 16:01:00,0 days 00:00:00,294.400000,196.000000,NaN,Norway


In [30]:
customer_features["ReturnInvoices"] = customer_features["ReturnInvoices"].fillna(0).astype(int)
customer_features["ReturnRate"] = (
    customer_features["ReturnInvoices"] /
    (customer_features["Frequency"] + customer_features["ReturnInvoices"])
)
customer_features.head()

,Recency,Frequency,Monetary,TotalQuantity,UniqueProducts,AvgUnitPrice,AvgBasketValue,FirstPurchase,LastPurchase,TenureDays,AvgOrderValue,AvgOrderItems,ReturnInvoices,Country,ReturnRate
CustomerID,,,,,,,,,,,,,,,
12346,326,1,77183.60,74215,1,1.040000,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00,0 days 00:00:00,77183.600000,74215.000000,1,United Kingdom,0.5
12347,2,7,4310.00,2458,103,2.644011,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00,365 days 00:55:00,615.714286,351.142857,0,Iceland,0.0
12348,75,4,1437.24,2332,21,0.692963,53.231111,2010-12-16 19:09:00,2011-09-25 13:13:00,282 days 18:04:00,359.310000,583.000000,0,Finland,0.0
12349,19,1,1457.55,630,72,4.237500,20.243750,2011-11-21 09:51:00,2011-11-21 09:51:00,0 days 00:00:00,1457.550000,630.000000,0,Italy,0.0
12350,310,1,294.40,196,16,1.581250,18.400000,2011-02-02 16:01:00,2011-02-02 16:01:00,0 days 00:00:00,294.400000,196.000000,0,Norway,0.0


In [31]:
for col in["Recency", "Frequency", "Monetary",
          "TotalQuantity", "UniqueProducts", "AvgOrderValue"]:
    customer_features[f'log_{col}'] = np.log1p(customer_features[col])

customer_features = customer_features.reset_index()
customer_features

,CustomerID,Recency,Frequency,Monetary,TotalQuantity,UniqueProducts,AvgUnitPrice,AvgBasketValue,FirstPurchase,LastPurchase,...,AvgOrderItems,ReturnInvoices,Country,ReturnRate,log_Recency,log_Frequency,log_Monetary,log_TotalQuantity,log_UniqueProducts,log_AvgOrderValue
0,12346,326,1,77183.60,74215,1,1.040000,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00,...,74215.000000,1,United Kingdom,0.500000,5.789960,0.693147,11.253955,11.214735,0.693147,11.253955
1,12347,2,7,4310.00,2458,103,2.644011,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00,...,351.142857,0,Iceland,0.000000,1.098612,2.079442,8.368925,7.807510,4.644391,6.424406
2,12348,75,4,1437.24,2332,21,0.692963,53.231111,2010-12-16 19:09:00,2011-09-25 13:13:00,...,583.000000,0,Finland,0.000000,4.330733,1.609438,7.271175,7.754910,3.091042,5.886965
3,12349,19,1,1457.55,630,72,4.237500,20.243750,2011-11-21 09:51:00,2011-11-21 09:51:00,...,630.000000,0,Italy,0.000000,2.995732,0.693147,7.285198,6.447306,4.290459,7.285198
4,12350,310,1,294.40,196,16,1.581250,18.400000,2011-02-02 16:01:00,2011-02-02 16:01:00,...,196.000000,0,Norway,0.000000,5.739793,0.693147,5.688330,5.283204,2.833213,5.688330
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4329,18280,278,1,180.60,45,10,4.765000,18.060000,2011-03-07 09:52:00,2011-03-07 09:52:00,...,45.000000,0,United Kingdom,0.000000,5.631212,0.693147,5.201806,3.828641,2.397895,5.201806
4330,18281,181,1,80.82,54,7,5.622857,11.545714,2011-06-12 10:53:00,2011-06-12 10:53:00,...,54.000000,0,United Kingdom,0.000000,5.204007,0.693147,4.404522,4.007333,2.079442,4.404522
4331,18282,8,2,178.05,103,12,5.199167,14.837500,2011-08-05 13:35:00,2011-12-02 11:43:00,...,51.500000,1,United Kingdom,0.333333,2.197225,1.098612,5.187665,4.644391,2.564949,4.500087
4332,18283,4,16,2039.58,1355,262,1.625007,2.836690,2011-01-06 14:14:00,2011-12-06 12:02:00,...,84.687500,0,United Kingdom,0.000000,1.609438,2.833213,7.620989,7.212294,5.572154,4.855725


In [80]:
# Save
# ---------------------------------------------------------------------------
import os
out_dir = Path.cwd().parent
os.makedirs(out_dir / "data", exist_ok=True)
df.to_csv(out_dir/ "data" / "online_retail_clean.csv", index=False)
customer_features.to_csv(out_dir / "data" / "customer_features.csv", index=False)

print("\nSaved:")
print(" ", out_dir / "data" / "online_retail_clean.csv", df.shape)
print(" ", out_dir / "data" / "customer_features.csv", customer_features.shape)

print("\nCustomer feature preview:")
print(customer_features.head())
print("\nSummary stats:")
print(customer_features[["Recency", "Frequency", "Monetary",
                         "AvgOrderValue", "TenureDays", "ReturnRate"]]
      .describe().round(2))



Saved:
  /mnt/c/Users/anish/CSP_571_FinalProj/data/online_retail_clean.csv (391150, 14)
  /mnt/c/Users/anish/CSP_571_FinalProj/data/customer_features.csv (4334, 22)

Customer feature preview:
   CustomerID  Recency  Frequency  Monetary  TotalQuantity  UniqueProducts  \
0       12346      326          1  77183.60          74215               1   
1       12347        2          7   4310.00           2458             103   
2       12348       75          4   1437.24           2332              21   
3       12349       19          1   1457.55            630              72   
4       12350      310          1    294.40            196              16   

   AvgUnitPrice  AvgBasketValue       FirstPurchase        LastPurchase  ...  \
0      1.040000    77183.600000 2011-01-18 10:01:00 2011-01-18 10:01:00  ...   
1      2.644011       23.681319 2010-12-07 14:57:00 2011-12-07 15:52:00  ...   
2      0.692963       53.231111 2010-12-16 19:09:00 2011-09-25 13:13:00  ...   
3      4.237500   